
# 🧠 Sports Betting Fraud Detection

**Group Members:** Aryaman Arora (aa833), Hyunjin Lee (hl494), Suhaib Mansour (sm1097), Emily Zhao (egz5)

This Jupyter Notebook implements the data collection, cleaning, feature engineering, visualization, and machine learning pipeline for detecting suspicious betting activity — as described in our project proposal.


In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import difflib
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


## 🧹 Step 1: Load and Clean Data

In [ ]:

def standardize_team_names(df, col, canonical_names, cutoff=0.7):
    def match_name(name):
        matches = difflib.get_close_matches(str(name), canonical_names, n=1, cutoff=cutoff)
        return matches[0] if matches else name
    df[col] = df[col].apply(match_name)
    return df

def clean_betting_data(match_df, odds_df):
    match_df['Date'] = pd.to_datetime(match_df['Date'], errors='coerce')
    odds_df['Date'] = pd.to_datetime(odds_df['Date'], errors='coerce')
    match_df = match_df.dropna(subset=['HomeTeam', 'AwayTeam'])
    match_df = match_df.drop_duplicates(subset=['HomeTeam', 'AwayTeam', 'Date'])
    merged = pd.merge(match_df, odds_df, on=['Date', 'HomeTeam', 'AwayTeam'], how='inner')
    merged['delta_odds'] = merged['ClosingOdds'] - merged['OpeningOdds']
    merged['odds_volatility'] = merged[['OpeningOdds', 'ClosingOdds']].std(axis=1)
    merged['odds_pct_change'] = (merged['delta_odds'] / merged['OpeningOdds']).fillna(0)
    numeric_cols = ['delta_odds', 'odds_volatility', 'odds_pct_change']
    merged[numeric_cols] = merged[numeric_cols].fillna(merged[numeric_cols].median())
    return merged


## ⚙️ Step 2: Feature Scaling and Baseline Labels

In [ ]:

def add_features_and_labels(df):
    scaler = MinMaxScaler()
    features = ['delta_odds', 'odds_volatility', 'odds_pct_change']
    df[[f + '_scaled' for f in features]] = scaler.fit_transform(df[features])
    threshold = df['delta_odds'].abs().quantile(0.95)
    df['suspicious'] = (df['delta_odds'].abs() >= threshold).astype(int)
    return df


## 📊 Step 3: Visualization

In [ ]:

def plot_distributions(df):
    plt.figure(figsize=(8, 5))
    sns.histplot(df['delta_odds'], bins=30, kde=True)
    plt.title("Distribution of Odds Change (ΔOdds)")
    plt.xlabel("ΔOdds")
    plt.ylabel("Frequency")
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.boxplot(x='suspicious', y='odds_volatility', data=df)
    plt.title("Volatility by Suspicious Label")
    plt.show()


## 🤖 Step 4: Baseline Machine Learning Model

In [ ]:

def train_baseline_model(df):
    X = df[['delta_odds', 'odds_volatility', 'odds_pct_change']]
    y = df['suspicious']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("=== Random Forest Baseline Model ===")
    print(classification_report(y_test, y_pred))
    feature_importances = pd.Series(model.feature_importances_, index=X.columns)
    feature_importances.plot(kind='bar', title='Feature Importances')
    plt.show()
    return model


## 🚀 Step 5: Run the Full Pipeline

In [ ]:

# Demo dataset (replace with your own CSVs for real analysis)
match_df = pd.DataFrame({
    'Date': pd.date_range('2023-08-01', periods=10, freq='D'),
    'HomeTeam': ['TeamA']*5 + ['TeamB']*5,
    'AwayTeam': ['TeamC']*5 + ['TeamD']*5,
    'FTHG': np.random.randint(0,5,10),
    'FTAG': np.random.randint(0,5,10)
})

odds_df = pd.DataFrame({
    'Date': pd.date_range('2023-08-01', periods=10, freq='D'),
    'HomeTeam': ['TeamA']*5 + ['TeamB']*5,
    'AwayTeam': ['TeamC']*5 + ['TeamD']*5,
    'OpeningOdds': np.random.uniform(1.5, 3.5, 10),
    'ClosingOdds': np.random.uniform(1.5, 3.5, 10)
})

merged = clean_betting_data(match_df, odds_df)
merged = add_features_and_labels(merged)
plot_distributions(merged)
train_baseline_model(merged)
merged.to_csv("Cleaned_Betting_Data.csv", index=False)
print("✅ Cleaned dataset saved as Cleaned_Betting_Data.csv")
